# Sign Language Translation: ASL -> English

End-to-end notebook for the **How2Sign** translation project.  
Architecture: ResNet-18 visual encoder + landmark spatial attention -> Temporal Transformer -> Decoder (BPE, vocab=8000)

**Sections**
1. Environment check  
2. Configuration  
3. Tokenizer  
4. Data pipeline  
5. Model architecture  
6. Training (launch / resume)  
7. Training log analysis  
8. Evaluation metrics  
9. Inference demo  
10. LLM-as-Judge (Claude)  

## 1. Environment Check

In [ ]:
# av must be imported before torch on Windows to avoid a segfault
# when the datasets library streams Video-feature data.
import av  # noqa: F401

import sys, os
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")
print(f"Device      : {device}")
if device.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Project root: {PROJECT_ROOT}")

## 2. Configuration

All hyperparameters live in `training/config.py`. Nothing is hardcoded elsewhere.

In [ ]:
from training.config import (
    # Paths
    ROOT, CHECKPOINT_DIR, DATA_CACHE_DIR, TOKENIZER_MODEL,
    # Dataset
    DATASET_NAME, DATASET_SPLITS,
    # Video
    IMG_SIZE, MAX_FRAMES, IMAGENET_MEAN, IMAGENET_STD,
    # Landmarks
    N_LANDMARKS, HEATMAP_SIGMA,
    # Tokenizer
    VOCAB_SIZE, PAD_ID, BOS_ID, EOS_ID, UNK_ID, MAX_TEXT_LEN,
    # Model
    D_MODEL, N_HEADS, DIM_FEEDFORWARD, N_ENC_LAYERS, N_DEC_LAYERS, DROPOUT,
    # Training
    MICRO_BATCH_SIZE, GRAD_ACCUM_STEPS, GRAD_CLIP_NORM, LABEL_SMOOTHING,
    WEIGHT_DECAY, PHASE1_EPOCHS, PHASE1_LR, PHASE2_EPOCHS, PHASE2_LR,
    TOTAL_EPOCHS, CHECKPOINT_EVERY, USE_AMP,
    # Inference
    BEAM_SIZE,
    # Evaluation
    LLM_JUDGE_MODEL, LLM_JUDGE_SAMPLES,
)

print("=== Model Architecture ===")
print(f"  d_model        : {D_MODEL}")
print(f"  n_heads        : {N_HEADS}")
print(f"  enc layers     : {N_ENC_LAYERS}")
print(f"  dec layers     : {N_DEC_LAYERS}")
print(f"  ff dim         : {DIM_FEEDFORWARD}")
print(f"  vocab size     : {VOCAB_SIZE}")
print()
print("=== Training ===")
print(f"  Phase 1        : {PHASE1_EPOCHS} epochs, lr={PHASE1_LR}, backbone frozen")
print(f"  Phase 2        : {PHASE2_EPOCHS} epochs, lr={PHASE2_LR}, backbone unfrozen")
print(f"  Effective batch: {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS}  (micro={MICRO_BATCH_SIZE} x accum={GRAD_ACCUM_STEPS})")
print(f"  AMP fp16       : {USE_AMP}")
print()
print("=== Paths ===")
print(f"  Checkpoints    : {CHECKPOINT_DIR}")
print(f"  Tokenizer      : {TOKENIZER_MODEL}")

## 3. Tokenizer

SentencePiece BPE tokenizer (vocab=8000).  
Special tokens: PAD=0, BOS=1, EOS=2, UNK=3

In [ ]:
from data.tokenizer import SignTokenizer

tokenizer = SignTokenizer()
if TOKENIZER_MODEL.exists():
    tokenizer.load(TOKENIZER_MODEL)
else:
    print("Tokenizer model not found -- run training/train.py first or train here:")
    print("  from data.download import load_dataset_split")
    print("  ds = load_dataset_split('train', streaming=True)")
    print("  texts = [ex['translation'] for ex in ds.take(3000)]")
    print("  tokenizer.train(texts)")

In [ ]:
# Tokenizer round-trip test
if tokenizer._sp is not None:
    sample = "The man is explaining how to cook pasta."
    ids = tokenizer.encode(sample, add_bos=True, add_eos=True)
    decoded = tokenizer.decode(ids, skip_special=True)
    print(f"Input   : {sample}")
    print(f"Token IDs ({len(ids)}): {ids[:15]} ...")
    print(f"Decoded : {decoded}")
    print(f"Vocab size: {tokenizer.vocab_size}")

## 4. Data Pipeline

### 4.1 Loading the dataset (streaming)

Dataset: `bdanko/how2sign-rgb-front-clips` on HuggingFace  
- ~31K training clips, ~1.7K validation clips  
- Fields: `mp4` (raw video bytes), `json` (SENTENCE text)

In [ ]:
from data.download import load_dataset_split

# Stream a few samples to inspect
ds = load_dataset_split("train", streaming=True)
samples = list(ds.take(3))

for i, s in enumerate(samples):
    video_bytes = s["video"]
    text = s["translation"]
    print(f"[{i}] text  : {text}")
    print(f"     video : {len(video_bytes):,} bytes")

### 4.2 Frame extraction

Decoded frames cached to `data/cache/frames/<sha1>.pt`.  
First call: full PyAV decode (~300-600 ms). Every subsequent call: disk load (~1-5 ms).

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
from data.download import extract_frames
from training.config import IMAGENET_MEAN, IMAGENET_STD

video_bytes = samples[0]["video"]

# Cold decode (writes cache on first run)
t0 = time.perf_counter()
frames, frame_mask = extract_frames(video_bytes, max_frames=MAX_FRAMES, img_size=IMG_SIZE)
cold_ms = (time.perf_counter() - t0) * 1000

# Cache hit
t0 = time.perf_counter()
frames, frame_mask = extract_frames(video_bytes, max_frames=MAX_FRAMES, img_size=IMG_SIZE)
hot_ms = (time.perf_counter() - t0) * 1000

n_real = int(frame_mask.sum())
print(f"Frames      : {n_real} real  (padded to {MAX_FRAMES})")
print(f"Shape       : {frames.shape}  {frames.dtype}")
print(f"Cold decode : {cold_ms:.0f} ms")
print(f"Cache hit   : {hot_ms:.1f} ms  ({cold_ms / max(hot_ms, 0.01):.0f}x faster)")

# Denormalise and display 8 evenly-spaced frames
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

n_show = min(8, n_real)
idxs = np.linspace(0, n_real - 1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for ax, idx in zip(axes, idxs):
    img = (frames[idx] * std + mean).permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img)
    ax.set_title(f"f{idx}", fontsize=8)
    ax.axis("off")

plt.suptitle(f'Clip: "{samples[0]["translation"]}"', fontsize=10)
plt.tight_layout()
plt.show()

### 4.3 Dataset classes

- `How2SignDataset` -- map-style, for validation / overfit tests (loads all records into RAM)  
- `How2SignStreamingDataset` -- iterable, for full training (lazy streaming + shuffle buffer)

In [ ]:
from data.dataset import How2SignDataset, How2SignStreamingDataset, collate_fn
from torch.utils.data import DataLoader

# Quick smoke test with 5 samples
records = list(load_dataset_split("train", streaming=True).take(5))
ds_test = How2SignDataset(records, tokenizer, extract_lm=False)

item = ds_test[0]
print("Item keys    :", list(item.keys()))
print("frames       :", item["frames"].shape, item["frames"].dtype)
print("heatmaps     :", item["heatmaps"].shape)
print("token_ids    :", item["token_ids"].shape, item["token_ids"][:10].tolist())
print("frame_mask   :", item["frame_mask"].shape, int(item["frame_mask"].sum()), "real frames")
print("text         :", item["text"])

# Batch
loader = DataLoader(ds_test, batch_size=2, collate_fn=collate_fn)
batch = next(iter(loader))
print("\nBatch frames :", batch["frames"].shape)

## 5. Model Architecture

```
Video (B, T, 3, 256, 256)
    |
    v
VisualEncoder
  ResNet-18 backbone  -> (B*T, 512, 8, 8)
  SpatialAttention    -> landmark heatmap gates feature maps
  AdaptiveAvgPool + Linear projection -> (B, T, d_model=512)
    |
    v
TemporalEncoder
  Sinusoidal PE + 4-layer TransformerEncoder -> (B, T, 512)
    |
    v
TextDecoder
  Token + positional embeddings
  4-layer TransformerDecoder (cross-attends to encoder output)
  Output projection (weight-tied with token emb) -> (B, S, vocab=8000)
```

In [ ]:
from models.translator import SignLanguageTranslator

model = SignLanguageTranslator(vocab_size=tokenizer.vocab_size)
params = model.count_parameters()

print("Parameter counts:")
for k, v in params.items():
    print(f"  {k:<20}: {v:>12,}")

In [ ]:
# Forward pass sanity check (no grad, CPU)
import torch

B, T, S = 2, 8, 12
frames_dummy    = torch.randn(B, T, 3, IMG_SIZE, IMG_SIZE)
tgt_dummy       = torch.randint(1, 100, (B, S))
frame_mask_dummy = torch.ones(B, T, dtype=torch.bool)

model.eval()
with torch.no_grad():
    logits = model(frames_dummy, tgt_dummy, frame_mask=frame_mask_dummy)

print(f"Input  frames: {frames_dummy.shape}")
print(f"Input  tgt   : {tgt_dummy.shape}")
print(f"Output logits: {logits.shape}  (should be {B} x {S} x {tokenizer.vocab_size})")

### 5.1 Sub-module inspection

In [ ]:
from models.visual_encoder import VisualEncoder, SpatialAttention
from models.temporal_encoder import TemporalEncoder, SinusoidalPositionalEncoding
from models.text_decoder import TextDecoder

print("VisualEncoder:")
print(f"  backbone channels : 512")
print(f"  output d_model    : {D_MODEL}")
print()
print("TemporalEncoder:")
print(f"  n_layers          : {N_ENC_LAYERS}")
print(f"  n_heads           : {N_HEADS}")
print(f"  dim_feedforward   : {DIM_FEEDFORWARD}")
print()
print("TextDecoder:")
print(f"  n_layers          : {N_DEC_LAYERS}")
print(f"  vocab_size        : {tokenizer.vocab_size}")
print(f"  max_len           : {MAX_TEXT_LEN}")
print(f"  weight tying      : token_emb = output_proj.weight")

## 6. Training

### 6.1 Trainer overview

| Phase | Epochs | LR | Backbone |
|-------|--------|----|----------|
| 1 | 1-15 | 3e-4 | Frozen |
| 2 | 16-30 | 3e-5 | Unfrozen |

Loss: CrossEntropyLoss + label_smoothing=0.1  
Optimizer: AdamW (weight_decay=0.01)  
Scheduler: CosineAnnealingLR  
Gradient accumulation: 8 steps (effective batch = 32)

In [ ]:
# === OVERFIT SANITY CHECK (100 clips, 30 epochs) ===
# Run this cell to verify everything works before committing to full training.
# Expected: loss drops from ~9 to ~5, best BLEU ~0.3

# Uncomment to run:

# import subprocess, sys
# result = subprocess.run(
#     [sys.executable, "training/train.py", "--no-wandb", "--overfit-test", "--no-landmarks"],
#     cwd=str(PROJECT_ROOT),
# )
print("Overfit test: uncomment above and run to verify pipeline end-to-end.")

In [ ]:
# === FULL TRAINING (streams from HuggingFace, ~2 days on RTX 4090) ===
# Runs in background; tail training_full.log to monitor.

# Uncomment to launch:

# import subprocess, sys
# proc = subprocess.Popen(
#     [sys.executable, "training/train.py", "--no-wandb", "--no-landmarks"],
#     cwd=str(PROJECT_ROOT),
#     stdout=open(PROJECT_ROOT / "training_full.log", "w"),
#     stderr=subprocess.STDOUT,
# )
# print(f"Training PID: {proc.pid}  -- tail training_full.log to monitor")
print("Full training: uncomment above to launch (or use terminal).")
print("Checkpoint saves: every 5 epochs + best BLEU -> checkpoints/")

### 6.2 Resuming from checkpoint

In [ ]:
# List available checkpoints
checkpoints = sorted(CHECKPOINT_DIR.glob("*.pt"))
if checkpoints:
    print("Available checkpoints:")
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / 1e6
        print(f"  {ckpt.name:<35} {size_mb:.1f} MB")
else:
    print("No checkpoints found -- train the model first.")

## 7. Training Log Analysis

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

log_path = PROJECT_ROOT / "training_full.log"
if not log_path.exists():
    log_path = PROJECT_ROOT / "training_overfit.log"

epoch_re = re.compile(
    r"Epoch (\d+)/(\d+)\s+train_loss=([\d.]+)\s+val_loss=([\d.]+)\s+BLEU=([\d.]+)\s+lr=([\d.e+-]+)"
)

epochs, train_losses, val_losses, bleus, lrs = [], [], [], [], []

if log_path.exists():
    with open(log_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = epoch_re.search(line)
            if m:
                epochs.append(int(m.group(1)))
                train_losses.append(float(m.group(3)))
                val_losses.append(float(m.group(4)))
                bleus.append(float(m.group(5)))
                lrs.append(float(m.group(6)))

    print(f"Parsed {len(epochs)} epoch entries from {log_path.name}")
    if epochs:
        print(f"  Latest epoch  : {epochs[-1]}")
        print(f"  Train loss    : {train_losses[0]:.4f} -> {train_losses[-1]:.4f}")
        print(f"  Val loss      : {val_losses[0]:.4f} -> {val_losses[-1]:.4f}")
        print(f"  Best BLEU     : {max(bleus):.4f} (epoch {epochs[bleus.index(max(bleus))]})") 
else:
    print(f"Log file not found: {log_path}")
    print("Run training first, then re-execute this cell.")

In [ ]:
if epochs:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curves
    ax = axes[0]
    ax.plot(epochs, train_losses, label="Train", marker="o", markersize=3)
    ax.plot(epochs, val_losses,   label="Val",   marker="s", markersize=3)
    if PHASE1_EPOCHS < max(epochs):
        ax.axvline(PHASE1_EPOCHS, color="gray", linestyle="--", alpha=0.7, label="Phase 2 start")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title("Loss Curves")
    ax.legend()
    ax.grid(alpha=0.3)

    # BLEU curve
    ax = axes[1]
    ax.plot(epochs, bleus, color="green", marker="o", markersize=3, label="BLEU-4")
    if PHASE1_EPOCHS < max(epochs):
        ax.axvline(PHASE1_EPOCHS, color="gray", linestyle="--", alpha=0.7)
    ax.axhline(10, color="red", linestyle=":", alpha=0.7, label="Target (BLEU=10)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BLEU-4")
    ax.set_title("Validation BLEU-4")
    ax.legend()
    ax.grid(alpha=0.3)

    # LR schedule
    ax = axes[2]
    ax.plot(epochs, lrs, color="orange", marker="o", markersize=3)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning Rate")
    ax.set_title("LR Schedule (Cosine)")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1e"))
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved -> training_curves.png")
else:
    print("No epoch data to plot yet.")

## 8. Evaluation Metrics

Metrics computed after training completes:  
- **BLEU-4** (sacrebleu, smoothing=exp)  
- **ROUGE-L** (rouge-score)  
- **METEOR** (nltk)

In [ ]:
from evaluation.metrics import compute_all, print_results

# Demo with synthetic predictions (replace with real model outputs after training)
demo_hypotheses = [
    "the man is showing how to cook",
    "she is signing about the weather today",
    "he talks about his family",
]
demo_references = [
    "the man demonstrates how to prepare food",
    "she is describing the weather outside",
    "he is discussing his family members",
]

metrics = compute_all(demo_hypotheses, demo_references)
print_results(metrics, n_samples=len(demo_hypotheses))

### 8.1 Full evaluation on test split

In [ ]:
def evaluate_checkpoint(checkpoint_path, n_samples=200, beam_size=1):
    """
    Load a checkpoint and run evaluation on the test split.
    
    Args:
        checkpoint_path : path to a .pt checkpoint file
        n_samples       : number of test clips to evaluate
        beam_size       : 1=greedy, 4=beam search
    """
    import torch
    from data.download import load_dataset_split
    from data.dataset import How2SignDataset, collate_fn
    from models.translator import SignLanguageTranslator
    from evaluation.metrics import compute_all, print_results
    from torch.utils.data import DataLoader

    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        print(f"Checkpoint not found: {checkpoint_path}")
        return

    print(f"Loading checkpoint: {checkpoint_path.name}")
    ckpt = torch.load(checkpoint_path, map_location=device)

    model_eval = SignLanguageTranslator(vocab_size=tokenizer.vocab_size)
    model_eval.load_state_dict(ckpt["model_state_dict"])
    model_eval.to(device)
    model_eval.eval()
    print(f"  Epoch: {ckpt.get('epoch', '?')+1}  Best BLEU so far: {ckpt.get('best_bleu', '?'):.4f}")

    print(f"Loading {n_samples} test samples...")
    test_records = list(load_dataset_split("test", streaming=True).take(n_samples))
    test_ds = How2SignDataset(test_records, tokenizer, extract_lm=False)
    test_loader = DataLoader(test_ds, batch_size=8, collate_fn=collate_fn, shuffle=False)

    hypotheses, references = [], []
    with torch.no_grad():
        for batch in test_loader:
            frames     = batch["frames"].to(device)
            heatmaps   = batch["heatmaps"].to(device)
            frame_mask = batch["frame_mask"].to(device)

            preds = model_eval.translate(
                frames, tokenizer,
                heatmaps=heatmaps,
                frame_mask=frame_mask,
                beam_size=beam_size,
            )
            hypotheses.extend(preds)
            references.extend(batch["text"])

    metrics = compute_all(hypotheses, references)
    print_results(metrics, n_samples=len(hypotheses))
    return hypotheses, references, metrics


# Usage (uncomment after training):
# hyps, refs, metrics = evaluate_checkpoint(CHECKPOINT_DIR / "checkpoint_best.pt", n_samples=500, beam_size=4)
print("evaluate_checkpoint() defined. Call it once training completes.")

## 9. Inference Demo

Load the best checkpoint and translate a sample clip.

In [ ]:
best_ckpt = CHECKPOINT_DIR / "checkpoint_best.pt"

if best_ckpt.exists():
    print(f"Loading {best_ckpt.name} ...")
    ckpt = torch.load(best_ckpt, map_location=device)

    model_inf = SignLanguageTranslator(vocab_size=tokenizer.vocab_size)
    model_inf.load_state_dict(ckpt["model_state_dict"])
    model_inf.to(device)
    model_inf.eval()
    print(f"  Epoch           : {ckpt.get('epoch', '?') + 1}")
    print(f"  Best BLEU saved : {ckpt.get('best_bleu', 0.0):.4f}")
else:
    print("No checkpoint found yet. Run training first.")
    model_inf = None

In [ ]:
if model_inf is not None:
    # Grab a validation clip and translate it
    from data.download import extract_frames

    val_sample = list(load_dataset_split("val", streaming=True).take(1))[0]
    video_bytes = val_sample["video"]
    reference   = val_sample["translation"]

    frames, frame_mask = extract_frames(video_bytes, max_frames=MAX_FRAMES, img_size=IMG_SIZE)
    frames_batch     = frames.unsqueeze(0).to(device)      # (1, T, 3, H, W)
    frame_mask_batch = frame_mask.unsqueeze(0).to(device)  # (1, T)

    with torch.no_grad():
        # Greedy
        pred_greedy = model_inf.translate(
            frames_batch, tokenizer,
            frame_mask=frame_mask_batch, beam_size=1
        )[0]
        # Beam search
        pred_beam = model_inf.translate(
            frames_batch, tokenizer,
            frame_mask=frame_mask_batch, beam_size=BEAM_SIZE
        )[0]

    print(f"Reference    : {reference}")
    print(f"Greedy       : {pred_greedy}")
    print(f"Beam (k={BEAM_SIZE})    : {pred_beam}")
else:
    print("Load a checkpoint first (cell above).")

## 10. LLM-as-Judge (Claude)

Uses `claude-sonnet-4-6` to score hypotheses on adequacy, fluency, and meaning (1-5 each).  
Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
from evaluation.llm_judge import judge_batch, summarise, print_examples, pearson_correlation, per_sample_bleu

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
if api_key:
    print("ANTHROPIC_API_KEY found -- LLM judge ready.")
else:
    print("ANTHROPIC_API_KEY not set.")
    print("Set it with: os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")
    print("Then re-run this cell.")

In [ ]:
# Run LLM judge on 200 test samples (requires ANTHROPIC_API_KEY and a trained model)
# Uncomment after training completes.

# if api_key and model_inf is not None:
#     hyps, refs, _ = evaluate_checkpoint(best_ckpt, n_samples=LLM_JUDGE_SAMPLES)
#
#     judge_scores = judge_batch(
#         hyps, refs,
#         model=LLM_JUDGE_MODEL,
#         batch_size=10,
#         max_samples=LLM_JUDGE_SAMPLES,
#     )
#
#     summary = summarise(judge_scores)
#     print("\n=== LLM Judge Summary ===")
#     for k, v in summary.items():
#         print(f"  {k:<22}: {v:.4f}")
#
#     print_examples(judge_scores, n=20)
#
#     # Pearson correlation: LLM composite vs sentence-level BLEU
#     bleu_per_sample = per_sample_bleu(hyps, refs)
#     r = pearson_correlation(judge_scores, bleu_per_sample)
#     print(f"\nPearson r(LLM, BLEU) = {r:.4f}")
#
#     # Save scores
#     from evaluation.llm_judge import save_scores
#     save_scores(judge_scores, str(PROJECT_ROOT / "llm_judge_scores.json"))
print("LLM judge: uncomment above after training completes and API key is set.")

## Appendix: Checkpoint Comparison

In [ ]:
# Compare multiple checkpoints side-by-side on a small validation slice

def quick_bleu_for_checkpoint(ckpt_path, n_samples=50):
    """Return val BLEU for a single checkpoint (greedy, n_samples clips)."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = SignLanguageTranslator(vocab_size=tokenizer.vocab_size)
    m.load_state_dict(ckpt["model_state_dict"])
    m.to(device).eval()

    records = list(load_dataset_split("val", streaming=True).take(n_samples))
    ds = How2SignDataset(records, tokenizer, extract_lm=False)
    loader = DataLoader(ds, batch_size=8, collate_fn=collate_fn)

    from evaluation.metrics import compute_bleu
    hyps, refs = [], []
    with torch.no_grad():
        for batch in loader:
            preds = m.translate(
                batch["frames"].to(device), tokenizer,
                frame_mask=batch["frame_mask"].to(device), beam_size=1
            )
            hyps.extend(preds)
            refs.extend(batch["text"])

    return compute_bleu(hyps, refs)["bleu4"]


if checkpoints:
    print("Checkpoint comparison (uncomment to run -- takes a few minutes per ckpt):")
    for ckpt in checkpoints:
        print(f"  {ckpt.name}")
    # Uncomment to run:
    # for ckpt in checkpoints:
    #     bleu = quick_bleu_for_checkpoint(ckpt, n_samples=50)
    #     print(f"  {ckpt.name:<35} BLEU={bleu:.4f}")
else:
    print("No checkpoints to compare yet.")